In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
# from textblob import TextBlob
import time
import pandas as pd
import matplotlib.pyplot as plt
import re
import os
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from deep_translator import GoogleTranslator  # Gunakan deep-translator

# Setup Selenium WebDriver (Lazy Initialization)
chromedriver_path = "/opt/homebrew/bin/chromedriver"
service = Service(chromedriver_path)
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-extensions")
options.add_argument("--profile-directory=Default")
options.add_argument("--disable-infobars")
# options.add_argument("--headless")  # Uncomment untuk mode headless
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")

driver = None  # Global driver for lazy initialization

analyzer = SentimentIntensityAnalyzer()

def get_driver():
    """
    Menginisialisasi driver hanya saat diperlukan.
    """
    global driver
    if not driver:
        driver = webdriver.Chrome(service=service, options=options)
    return driver

def clean_comment(comment):
    """
    Membersihkan komentar dengan menghapus noise seperti URL, emoji, angka, dan karakter khusus.
    """
    comment = re.sub(r"http\S+|www\S+|https\S+", "", comment, flags=re.MULTILINE)  # Hapus URL
    comment = re.sub(r"\d+", "", comment)  # Hapus angka
    comment = re.sub(r"[^\w\s,]", "", comment)  # Hapus emoji dan karakter khusus
    comment = re.sub(r"\s+", " ", comment).strip()  # Hapus spasi ganda
    return comment

def translate_to_indonesian(comment):
    """
    Menerjemahkan komentar dari bahasa Bali ke bahasa Indonesia menggunakan GoogleTranslator.
    """
    try:
        translated = GoogleTranslator(source="auto", target="id").translate(comment)
        return translated
    except Exception as e:
        print(f"Error dalam penerjemahan: {e}")
        return comment  # Jika gagal, kembalikan komentar asli

def analyze_sentiment(comments):
    """
    Analisis sentimen untuk komentar bahasa Bali dengan terjemahan ke bahasa Indonesia.
    Menambahkan aturan khusus untuk mendeteksi kata tertentu sebagai sentimen negatif.
    """
    processed_comments = []  # Menyimpan komentar yang sudah diproses
    sentiments = []  # Menyimpan hasil sentimen

    for comment in comments:
        try:
            # Membersihkan komentar sebelum analisis
            cleaned_comment = clean_comment(comment)

            # Filter: hanya komentar dengan lebih dari 3 kata
            if len(cleaned_comment.split()) <= 3:
                continue

            # Terjemahkan komentar ke bahasa Indonesia
            translated_comment = translate_to_indonesian(cleaned_comment)
            processed_comments.append(translated_comment)

            # Menghitung skor sentimen dengan VADER
            scores = analyzer.polarity_scores(translated_comment)
            sentiment = "Netral"  # Default untuk netral

            if scores['compound'] > 0.05:
                sentiment = "Positif"
            elif scores['compound'] < -0.05:
                sentiment = "Negatif"

            sentiments.append(sentiment)

        except Exception as e:
            print(f"Error dalam analisis sentimen: {e}")
            processed_comments.append("")  # Tambahkan placeholder kosong
            sentiments.append("Netral")  # Sentimen default sebagai "Netral"
            
    return processed_comments, sentiments

def scrape_all_video_links(url):
    """
    Mengambil semua link video dari halaman TikTok pengguna dengan scrolling.
    """
    if not url:
        print("URL tidak valid. Silakan masukkan URL yang benar.")
        return []
    
    driver = get_driver()
    driver.get(url)
    print("Silakan selesaikan CAPTCHA jika muncul.")
    input("Tekan ENTER setelah Anda menyelesaikan CAPTCHA...")
    
    SCROLL_PAUSE_TIME = 3
    last_height = driver.execute_script("return document.body.scrollHeight")
    
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(SCROLL_PAUSE_TIME)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    video_links = [link["href"] for link in soup.find_all("a", href=True) if "/video/" in link["href"]]
    video_links = list(set(video_links))
    print(f"Jumlah video ditemukan: {len(video_links)}")
    return video_links

def scrape_tiktok_comments(url):
    """
    Mengambil semua komentar dari satu video TikTok dengan waktu tunggu.
    """
    if not url:
        print("URL tidak valid. Silakan masukkan URL yang benar.")
        return []

    try:
        driver = get_driver()
        driver.get(url)
        time.sleep(5)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        comment_elements = soup.find_all("span", class_="TUXText TUXText--tiktok-sans TUXText--weight-normal")
        comments = [element.get_text() for element in comment_elements]
        print(f"Jumlah komentar ditemukan di {url}: {len(comments)}")
        return comments
    except Exception as e:
        print(f"Terjadi error saat mengambil komentar: {e}")
        input("Tekan ENTER untuk mencoba lagi...")
        return scrape_tiktok_comments(url)

def load_video_links_from_csv(csv_file):
    """
    Memuat link video dari file CSV jika ada.
    """
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        return df["VideoLink"].tolist()
    return []

def save_video_links_to_csv(video_links, csv_file):
    """
    Menyimpan link video ke file CSV.
    """
    df = pd.DataFrame({"VideoLink": video_links})
    df.to_csv(csv_file, index=False)
    print(f"Link video disimpan ke '{csv_file}'.")

def load_processed_comments(csv_file):
    """
    Memuat komentar yang sudah diproses dari file CSV jika ada.
    """
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        return df
    return pd.DataFrame(columns=["VideoLink", "Comment"])

def save_comments_to_csv(df, csv_file):
    """
    Menyimpan komentar yang sudah diproses ke file CSV.
    """
    df.to_csv(csv_file, index=False)
    print(f"Komentar disimpan ke '{csv_file}'.")

def save_comments_to_csv(comments, csv_file):
    df = pd.DataFrame({"Comment": comments})
    df.to_csv(csv_file, index=False)
    print(f"Komentar disimpan ke '{csv_file}'.")

def start(tiktok_user_url, video_links_csv, comments_csv):
    print("Memeriksa file CSV untuk link video...")
    video_links = load_video_links_from_csv(video_links_csv)

    if not video_links:
        print("Tidak ada link video di file CSV. Mengambil link dari halaman pengguna TikTok...")
        video_links = scrape_all_video_links(tiktok_user_url)
        save_video_links_to_csv(video_links, video_links_csv)
    else:
        print(f"Memuat {len(video_links)} link video dari file CSV.")

    processed_df = load_processed_comments(comments_csv)
    processed_video_links = processed_df["VideoLink"].unique()

    for video_link in video_links:
        if video_link in processed_video_links:
            print(f"Komentar dari video {video_link} sudah diambil sebelumnya, melewati...")
            continue

        print(f"Mengambil komentar dari video: {video_link}")
        comments = scrape_tiktok_comments(video_link)
        
        if comments:
            temp_df = pd.DataFrame({
                "VideoLink": [video_link] * len(comments),
                "Comment": comments
            })
            processed_df = pd.concat([processed_df, temp_df], ignore_index=True)
            save_comments_to_csv(processed_df, comments_csv)
            
    print("Proses selesai. Komentar disimpan.")

def start_sentiment(sentiment_csv, comment_csv):
    """
    Proses analisis sentimen pada data komentar yang disimpan di file CSV.
    """
    processed_df = load_processed_comments(comment_csv)

    # Pastikan kolom "Comment" tidak mengandung NaN dan ubah menjadi string
    processed_df = processed_df.dropna(subset=["Comment"]).reset_index(drop=True)
    processed_df["Comment"] = processed_df["Comment"].astype(str)

    # Ambil daftar komentar
    comments = processed_df["Comment"].tolist()

    # Lakukan analisis sentimen
    processed_comments, sentiments = analyze_sentiment(comments)

    # Simpan hasil ke file CSV
    sentiment_df = pd.DataFrame({
        "VideoLink": processed_df["VideoLink"][:len(processed_comments)],  # Sesuaikan panjang
        "OriginalComment": processed_df["Comment"][:len(processed_comments)],
        "ProcessedComment": processed_comments,
        "Sentiment": sentiments
    })
    save_comments_to_csv(sentiment_df, sentiment_csv)
    
    print("Proses selesai. hasil sentimen disimpan.")


In [3]:
data = pd.read_csv("sentiment_koster.csv")
data

,VideoLink,OriginalComment,ProcessedComment,Sentiment
0,https://www.tiktok.com/@pasubayankostergiri/vi...,omon omon gen 😂😂😂 one comando lebih yakin 😂,omon omon gen one comando lebih yakin,Netral
1,https://www.tiktok.com/@pasubayankostergiri/vi...,11-13,Buleleng kapan maju kalau Wayan Ling Buleleng ...,Netral
2,https://www.tiktok.com/@pasubayankostergiri/vi...,133,wacana gen uluk ngalih suara the ambisui jdi B...,Netral
3,https://www.tiktok.com/@pasubayankostergiri/vi...,Buleleng pidan kal maju yan ? pocol gati Wayan...,"kami tidak butuh jalan tol pak,,",Netral
4,https://www.tiktok.com/@pasubayankostergiri/vi...,11-13,Mantap pak yan Mari sukseskan bali dengan MULI...,Negatif
...,...,...,...,...
521,https://www.tiktok.com/@pasubayankostergiri/vi...,3,Megae pak kesadaran diri sendiri to,Netral
522,https://www.tiktok.com/@pasubayankostergiri/vi...,menyala wiii 2 priode,"lucunya, editingnya masih gas ya om",Netral
523,https://www.tiktok.com/@pasubayankostergiri/vi...,10-29,kostergiri yes lanjutkan periode,Positif
524,https://www.tiktok.com/@pasubayankostergiri/vi...,3,"Rahayu Semua Baik, tapi saatnya satu jalur One...",Negatif


In [8]:
koster = pd.read_csv("processed_sentimen_koster.csv")
koster

,Unnamed: 0,ProcessedComment,Sentiment
0,0,omon omon gen one comando lebih yakin,Netral
1,1,Buleleng kapan maju kalau Wayan Ling Buleleng ...,Netral
2,2,wacana gen uluk ngalih suara the ambisui jdi B...,Netral
3,3,"kami tidak butuh jalan tol pak,,",Netral
4,4,Mantap pak yan Mari sukseskan bali dengan MULI...,Negatif
...,...,...,...
521,521,Megae pak kesadaran diri sendiri to,Netral
522,522,"lucunya, editingnya masih gas ya om",Netral
523,523,kostergiri yes lanjutkan periode,Positif
524,524,"Rahayu Semua Baik, tapi saatnya satu jalur One...",Negatif


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

def train_naive_bayes_model(df):
    """
    Train a Naive Bayes model using the ProcessedComment and Sentiment columns.
    """
    # Check required columns
    if 'ProcessedComment' not in df.columns or 'Sentiment' not in df.columns:
        raise ValueError("DataFrame must contain 'ProcessedComment' and 'Sentiment' columns.")

    # Extract features and labels
    comments = df['ProcessedComment'].astype(str).tolist()
    sentiments = df['Sentiment'].tolist()

    # Vectorize text data
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(comments)
    y = sentiments

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train a Multinomial Naive Bayes model
    model = MultinomialNB()
    model.fit(X_train, y_train)

    # Evaluate the model
    y_pred = model.predict(X_test)
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    return model, vectorizer

def analyze_with_naive_bayes(df, model, vectorizer):
    """
    Use a trained Naive Bayes model to analyze comments.
    """
    # Check if 'ProcessedComment' exists
    if 'ProcessedComment' not in df.columns:
        raise ValueError("DataFrame must contain 'ProcessedComment' column.")

    # Transform comments to the vectorized format
    comments = df['ProcessedComment'].astype(str).tolist()
    X = vectorizer.transform(comments)

    # Predict sentiments
    predictions = model.predict(X)
    df['NaiveBayesSentiment'] = predictions

    return df

